# This is a sample Jupyter Notebook

Below is an example of a code cell. 
Put your cursor into the cell and press Shift+Enter to execute it and select the next one, or click 'Run Cell' button.

Press Double Shift to search everywhere for classes, files, tool windows, actions, and settings.

To learn more about Jupyter Notebooks in PyCharm, see [help](https://www.jetbrains.com/help/pycharm/ipython-notebook-support.html).
For an overview of PyCharm, go to Help -> Learn IDE features or refer to [our documentation](https://www.jetbrains.com/help/pycharm/getting-started.html).

In [16]:
from hmmlearn.hmm import CategoricalHMM
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from prompt_toolkit.utils import to_str
from sklearn.metrics import confusion_matrix
import streamlit as st

import atexit
import time
import os

data = pd.read_csv("observable-data.csv")

In [17]:
data.drop(columns=["Region","Patpm","Weekday"], inplace=True)



In [18]:
prefix_to_symbol = {"41": 0, "42": 1, "43": 2, "44": 3}
symbol_to_prefix = {0: "41", 1: "42", 2: "43", 3: "44"}
n_symbols = 4
Hidden=[]

In [19]:
Tempo=[]
storage=[]
index=50
for value in data["Formation"]:
    temp=str(value)
    Tempo.append(temp)
while len(Tempo)>index:
    storage.append(Tempo[index])
    index=index+1
for value in storage:
    prefix = str(value)[:2]
    Hidden.append(prefix_to_symbol[prefix])


In [20]:
data.drop(index=[50,51,52,53,54,55,56,57,58,59,60,61,62,63], inplace=True)
data.reset_index(drop=True, inplace=True)

In [21]:
symbols = []
for value in data["Formation"]:
    prefix = str(value)[:2]
    symbols.append(prefix_to_symbol[prefix])
transmat_prior = np.array([
        [10.0, 10.0, 10.0, 10.0],
        [10.0, 10.0, 10.0, 10.0],
        [10.0, 10.0, 10.0, 10.0],
        [10.0, 10.0, 10.0, 10.0],
])

X = np.array(symbols).reshape(-1, 1)


In [22]:
model = CategoricalHMM(
    n_components=4,
    n_iter=200,
    random_state=0,
)

model.startprob_prior = np.array([1.0, 1.0, 1.0, 1.0])
model.transmat_prior  = transmat_prior
model.emissionprob_prior = np.ones((4, 4))
model.fit(X)
hidden_states = model.predict(X)
data["pred_hidden_state"] = hidden_states
data["pred_hidden_label"] = [symbol_to_prefix[s] for s in hidden_states]



In [23]:
last_state = symbols[-1]
print(last_state)
next_state_probs = model.transmat_[last_state]
print(next_state_probs)
next_state = int(np.argmax(next_state_probs))
print("Predicted next hidden state:", symbol_to_prefix[next_state])




0
[0.28059932 0.20288301 0.28013379 0.23638388]
Predicted next hidden state: 41


In [24]:
#sns.set_style("whitegrid")
#plt.plot(Hidden, label="Hidden State")
#plt.plot(hidden_states, label="Predicted Hidden State")
#plt.xlabel('timesteps')
#plt.ylabel('hidden state')
#plt.legend()
#model.score(X)
